In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# ----------------------------
# Create small dataset (20 samples)
# ----------------------------

data = []

for i in range(20):
    size = random.randint(50, 200)            # m^2
    bedrooms = random.randint(1, 5)
    zip_code = random.randint(0, 4)           # only 5 zip codes (0–4)
    wealth = random.randint(30, 80)

    # Simple price formula (so we know the ground truth)
    price = (
        2000 * size +
        10000 * bedrooms +
        5000 * wealth +
        15000 * zip_code
    )

    data.append([size, bedrooms, zip_code, wealth, price])

# Print dataset
for row in data:
    print(row)


[90, 1, 3, 77, 620000]
[124, 3, 2, 31, 463000]
[123, 5, 4, 78, 746000]
[58, 1, 0, 32, 286000]
[147, 5, 0, 45, 569000]
[149, 3, 1, 49, 588000]
[112, 3, 0, 34, 424000]
[73, 5, 1, 48, 451000]
[101, 2, 0, 40, 422000]
[144, 5, 1, 39, 548000]
[198, 2, 4, 50, 726000]
[96, 2, 1, 74, 597000]
[130, 4, 0, 55, 575000]
[140, 5, 4, 74, 760000]
[140, 1, 2, 46, 550000]
[167, 2, 4, 73, 779000]
[117, 3, 0, 40, 464000]
[160, 4, 1, 62, 685000]
[126, 5, 0, 68, 642000]
[100, 4, 1, 69, 600000]


In [2]:
# Split
train_data = data[:16]
test_data = data[16:]

print("Train size:", len(train_data))
print("Test size:", len(test_data))


Train size: 16
Test size: 4


In [3]:
def prepare_tensor(dataset):
    sizes = torch.tensor([x[0] for x in dataset], dtype=torch.float32).unsqueeze(1)
    bedrooms = torch.tensor([x[1] for x in dataset], dtype=torch.float32).unsqueeze(1)
    zip_codes = torch.tensor([x[2] for x in dataset], dtype=torch.long)
    wealth = torch.tensor([x[3] for x in dataset], dtype=torch.float32).unsqueeze(1)
    prices = torch.tensor([x[4] for x in dataset], dtype=torch.float32).unsqueeze(1)
    return sizes, bedrooms, zip_codes, wealth, prices

train_tensors = prepare_tensor(train_data)
test_tensors = prepare_tensor(test_data)


In [4]:
class HouseModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.zip_embedding = nn.Embedding(5, 3)  # 5 zip codes → 3D embedding
        
        self.fc = nn.Sequential(
            nn.Linear(3 + 3, 16),  # 3 numeric + 3 embedding
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, size, bedrooms, wealth, zip_code):
        
        zip_emb = self.zip_embedding(zip_code)
        
        x = torch.cat([size, bedrooms, wealth, zip_emb], dim=1)
        
        return self.fc(x)

model = HouseModel()


In [7]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

sizes, bedrooms, zip_codes, wealth, prices = train_tensors

for epoch in range(200):

    model.train()
    optimizer.zero_grad()

    outputs = model(sizes, bedrooms, wealth, zip_codes)
    loss = criterion(outputs, prices)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.2f}")


Epoch 0, Loss: 335700033536.00
Epoch 20, Loss: 334833385472.00
Epoch 40, Loss: 333889175552.00
Epoch 60, Loss: 332854886400.00
Epoch 80, Loss: 331723276288.00
Epoch 100, Loss: 330488938496.00
Epoch 120, Loss: 329147121664.00
Epoch 140, Loss: 327693500416.00
Epoch 160, Loss: 326123814912.00
Epoch 180, Loss: 324434264064.00


In [6]:
model.eval()

sizes, bedrooms, zip_codes, wealth, prices = test_tensors

with torch.no_grad():
    predictions = model(sizes, bedrooms, wealth, zip_codes)
    test_loss = criterion(predictions, prices)

print("Test Loss:", test_loss.item())

print("\nPredictions vs True:")
for pred, true in zip(predictions, prices):
    print(pred.item(), " | ", true.item())


Test Loss: 357632573440.0

Predictions vs True:
4693.27978515625  |  464000.0
6404.669921875  |  685000.0
5665.83154296875  |  642000.0
4890.90185546875  |  600000.0
